In [6]:

# --- Full example code demonstrating LlmAgent with Tools vs. Output Schema ---
import json # Needed for pretty printing dicts
import asyncio 
import os
from typing import List, Optional
from datetime import datetime   
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from pydantic import BaseModel, Field
from google.adk.models.lite_llm import LiteLlm # For multi-model support


In [ ]:
# --- 1. Define Constants ---
APP_NAME = "agent_comparison_app"
USER_ID = "test_user_456"
SESSION_ID_TOOL_AGENT = "session_tool_agent_xyz"
SESSION_ID_SCHEMA_AGENT = "session_schema_agent_xyz"
MODEL_NAME = "gpt-4o"

In [ ]:
# --- 2. Define Schemas ---

# Input schema used by both agents
class GuideInput(BaseModel):
    asset_field: str = Field(description="The field to get information about.")
    asset: str = Field(description="The asset to get information about.")

# Output schema ONLY for the second agent
class GuideOutput(BaseModel):
    field_Value_identifed: str = Field(description="field value captured by the agent")
    # Note: Population is illustrative; the LLM will infer or estimate this
    # as it cannot use tools when output_schema is set.

In [ ]:
# --- 3. Define the Tool (Updated for multiple fields) ---
def get_asset(asset_type: str, asset_field: str) -> dict:
    """Retrieves details for a specified asset type and field.

    Args:
        asset_type (str): The type of asset (e.g., "server", "pc_hmi").
        asset_field (str): The field of the asset to retrieve (e.g., "system_name", "type").

    Returns:
        dict: A dictionary containing the asset field information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'details' key with asset field description.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_asset called for asset_type: {asset_type}, asset_field: {asset_field} ---")  # Log tool execution
    asset_normalized = asset_type.lower().replace(" ", "_")
    field_normalized = asset_field.lower().replace(" ", "_")

    # Mock asset data with 3 fields each
    mock_asset_db = {
        "server": {
            "system_name": {
                "description": "Name given to the server, e.g., 'Factory Control Server'."
            },
            "type": {
                "description": "Indicates if the server is Physical (tangible hardware) or Virtual (VMware, Azure, etc.)."
            },
            "operating_system": {
                "description": "The OS running on the server, e.g., Windows, Linux, Ubuntu, VMware."
            }
        },
        "pc_hmi": {
            "system_name": {
                "description": "Name given to the PC-HMI device."
            },
            "type": {
                "description": "Indicates whether the PC-HMI is an Industrial PC or a Standard PC used for HMI tasks."
            },
            "operating_system": {
                "description": "The OS running on the PC-HMI, typically Windows or Linux-based for industrial apps."
            }
        }
    }

    # Check if asset exists
    if asset_normalized not in mock_asset_db:
        return {"status": "error", "error_message": f"Sorry, I don't have details for asset type '{asset_type}'."}

    # Check if field exists for the asset
    if field_normalized not in mock_asset_db[asset_normalized]:
        return {"status": "error", "error_message": f"Field '{asset_field}' not found for asset type '{asset_type}'."}

    # Return details
    return {"status": "success", "details": mock_asset_db[asset_normalized][field_normalized]}